In [ ]:
# Cell 1: Install dependencies
!pip install rank_bm25 sentence-transformers nltk pandas anthropic -q

In [13]:
# Cell 2: Imports
import pandas as pd
import numpy as np
import nltk
import json
import re
import textwrap
import anthropic
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, util

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

True

In [14]:
# Cell 4: Load dataset
df = pd.read_json("hf://datasets/zai-org/LongBench-v2/data.json")

print(f"Total examples: {len(df)}")
print(f"Domains:     {sorted(df['domain'].unique())}")
print(f"Sub-domains: {sorted(df['sub_domain'].unique())}")
print(f"Difficulty:  {sorted(df['difficulty'].unique())}")
print(f"Length:      {sorted(df['length'].unique())}")

Total examples: 503
Domains:     ['Code Repository Understanding', 'Long In-context Learning', 'Long Structured Data Understanding', 'Long-dialogue History Understanding', 'Multi-Document QA', 'Single-Document QA']
Sub-domains: ['Academic', 'Agent history QA', 'Code repo QA', 'Detective', 'Dialogue history QA', 'Event ordering', 'Financial', 'Governmental', 'Knowledge graph reasoning', 'Legal', 'Literary', 'Many-shot learning', 'Multi-news', 'New language translation', 'Table QA', 'User guide QA']
Difficulty:  ['easy', 'hard']
Length:      ['long', 'medium', 'short']


In [ ]:
# Cell 3: Config
DOMAIN      = "Single-Document QA"
SUB_DOMAIN  = "Academic"
DIFFICULTY  = "easy"
LENGTH      = "short"

TOP_K_SENTENCES = 3   # for BM25 / bi-encoder fallback

ANTHROPIC_API_KEY = ""

In [16]:
# Cell 5: Filter and load one sample
mask = (
    (df['domain'].str.lower()      == DOMAIN.lower()) &
    (df['difficulty'].str.lower()  == DIFFICULTY.lower()) &
    (df['length'].str.lower()      == LENGTH.lower())
)
if SUB_DOMAIN:
    mask &= (df['sub_domain'].str.lower() == SUB_DOMAIN.lower())

filtered = df[mask].reset_index(drop=True)
assert len(filtered) > 0, "No examples matched — try different parameters."

sample = filtered.iloc[0].to_dict()

answer_letter  = sample['answer']
answer_text    = sample[f'choice_{answer_letter}']
combined_query = f"{sample['question']} {answer_text}"

print(f"Loaded : {sample['_id']}")
print(f"Domain : {sample['domain']} / {sample['sub_domain']}")
print(f"Q      : {sample['question']}")
print(f"Answer ({answer_letter}): {answer_text}")

Loaded : 66f53892821e116aacb33225
Domain : Single-Document QA / Academic
Q      : Which of the following methods is used in this article to calculate the precision similarity of two documents?
Answer (B): For each word in the generated text, select the word in the reference text that is most similar to it, and calculate the maximum cosine similarity of the BERT hidden layer representation


In [17]:
# Cell 6: Sentence tokenization — shared by all methods
def split_into_sentences(text):
    sentences = nltk.sent_tokenize(text)
    return [s.strip() for s in sentences if len(s.strip()) >= 20]

context_sentences = split_into_sentences(sample['context'])
print(f"Total sentences in context: {len(context_sentences)}")

Total sentences in context: 684


In [18]:
# Cell 7: BM25 needle extraction (baseline, uses combined query)
def extract_needle_bm25(sentences, query, top_k=3):
    tokenized_corpus = [s.lower().split() for s in sentences]
    tokenized_query  = query.lower().split()
    bm25   = BM25Okapi(tokenized_corpus)
    scores = bm25.get_scores(tokenized_query)

    top_indices        = np.argsort(scores)[::-1][:top_k]
    top_indices_sorted = sorted(top_indices)
    return {
        "text":             " ".join(sentences[i] for i in top_indices_sorted),
        "sentence_indices": top_indices_sorted,
        "scores":           [round(float(scores[i]), 4) for i in top_indices_sorted],
        "method":           "BM25Okapi",
    }

bm25_result = extract_needle_bm25(context_sentences, combined_query, TOP_K_SENTENCES)
print("BM25 needle:")
print(textwrap.fill(bm25_result["text"], width=90))

BM25 needle:
2.2 EDIT-DISTANCE-BASED METRICS Several methods use word edit distance or word error rate
(Levenshtein, 1966), which quantify similarity using the number of edit operations
required to get from the candidate to the refer- ence. We use greedy matching to maximize
the matching similarity score,2 where each token is matched to the most similar token in
the other sentence. The cosine similarity of each word matching in PBERT are color-coded.


In [19]:
# Cell 8: Bi-encoder needle extraction (baseline, uses combined query)
def extract_needle_biencoder(sentences, query, model, top_k=3):
    query_emb     = model.encode(query, convert_to_tensor=True)
    sentence_embs = model.encode(sentences, convert_to_tensor=True,
                                 batch_size=64, show_progress_bar=False)
    scores = util.cos_sim(query_emb, sentence_embs)[0].cpu().numpy()

    top_indices        = np.argsort(scores)[::-1][:top_k]
    top_indices_sorted = sorted(top_indices)
    return {
        "text":             " ".join(sentences[i] for i in top_indices_sorted),
        "sentence_indices": top_indices_sorted,
        "scores":           [round(float(scores[i]), 4) for i in top_indices_sorted],
        "method":           "multi-qa-MiniLM-L6-cos-v1",
    }

print("Loading bi-encoder model...")
biencoder_model = SentenceTransformer("sentence-transformers/multi-qa-MiniLM-L6-cos-v1")
biencoder_result = extract_needle_biencoder(
    context_sentences, combined_query, biencoder_model, TOP_K_SENTENCES
)
print("Bi-encoder needle:")
print(textwrap.fill(biencoder_result["text"], width=90))

Loading bi-encoder model...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7033.64it/s]
BertModel LOAD REPORT from: sentence-transformers/multi-qa-MiniLM-L6-cos-v1
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Bi-encoder needle:
BERTSCORE computes the similarity of two sentences as a sum of cosine similarities between
their tokens’ embeddings. Given the reference x and candidate ˆ x, we compute BERT
embeddings and pairwise cosine similarity. Measuring the semantic similarity of texts.


In [ ]:
# Cell 9: LLM-guided needle extraction
#
# Design:
#   LLM's job  — read numbered sentences + Q&A, return ONLY indices (no reproduced text)
#   Code's job — use those indices to fetch exact strings from context_sentences
#
# The prompt intentionally prohibits the LLM from quoting or paraphrasing.
# It must output only a JSON object with a list of integer indices.

LLM_NEEDLE_PROMPT = """\
You are a precise evidence locator. You will be given:
  - A multiple-choice question
  - The correct answer (text, not the letter)
  - A numbered list of sentences from a long document

Your ONLY job is to identify which sentence numbers contain the evidence that
directly supports the correct answer. Evidence may be a single sentence or
spread across several non-consecutive sentences (distributed evidence).

Rules:
  - Output ONLY valid JSON. No explanation, no preamble, no markdown fences.
  - Do NOT reproduce, quote, or paraphrase any sentence text.
  - Return as few indices as possible to fully support the answer, at most 6 sentence indices, ordered ascending.
  - Return more indices only if the evidence is genuinely distributed.
  - If evidence is truly absent, return an empty list.

Output schema:
{{
  "indices": [<int>, ...],
  "reasoning": "<one short sentence explaining what the evidence shows — do NOT quote>"
}}

---
QUESTION: {question}
CORRECT ANSWER: {answer_text}

NUMBERED SENTENCES:
{numbered_sentences}
"""

def build_numbered_sentences(sentences):
    """Format sentences as  [0] text  [1] text  ... for the LLM prompt."""
    return "\n".join(f"[{i}] {s}" for i, s in enumerate(sentences))

def extract_needle_llm(sentences, question, answer_text, api_key, model="claude-haiku-4-5"):
    """
    Ask the LLM to return sentence indices only.
    Then fetch the exact strings from `sentences` using those indices.
    """
    client = anthropic.Anthropic(api_key=api_key)

    numbered = build_numbered_sentences(sentences)
    prompt   = LLM_NEEDLE_PROMPT.format(
        question=question,
        answer_text=answer_text,
        numbered_sentences=numbered,
    )

    response = client.messages.create(
        model=model,
        max_tokens=256,        # indices + one-line reasoning — should never need more
        messages=[{"role": "user", "content": prompt}],
    )

    raw = response.content[0].text.strip()

    # ── Parse JSON, tolerating minor whitespace/fence issues ──────────────────
    cleaned = re.sub(r"^```(?:json)?|```$", "", raw, flags=re.MULTILINE).strip()
    parsed  = json.loads(cleaned)

    indices   = sorted(set(int(i) for i in parsed.get("indices", [])
                           if 0 <= int(i) < len(sentences)))
    reasoning = parsed.get("reasoning", "")

    # ── Code fetches the exact strings — LLM never reproduced them ───────────
    needle_text = " ".join(sentences[i] for i in indices)

    return {
        "text":             needle_text,
        "sentence_indices": indices,
        "reasoning":        reasoning,
        "method":           f"LLM-guided ({model})",
        "raw_llm_output":   raw,
    }


llm_result = extract_needle_llm(
    sentences    = context_sentences,
    question     = sample['question'],
    answer_text  = answer_text,
    api_key      = ANTHROPIC_API_KEY,
)

print("LLM-GUIDED NEEDLE")
print("=" * 70)
print(f"Indices   : {llm_result['sentence_indices']}")
print(f"Reasoning : {llm_result['reasoning']}")
print(f"\nFetched text (exact strings from context):\n")
print(textwrap.fill(llm_result["text"], width=90))

LLM-GUIDED NEEDLE
Indices   : [114, 115, 117]
Reasoning : These sentences define precision (PBERT) as matching each token in the candidate to the most similar token in the reference using greedy matching and cosine similarity of contextual embeddings.

Fetched text (exact strings from context):

BERTSCORE The complete score matches each token in x to a token in ˆ x to compute recall,
and each token in ˆ x to a token in x to compute precision. We use greedy matching to
maximize the matching similarity score,2 where each token is matched to the most similar
token in the other sentence. For a reference x and candidate ˆ x, the recall, precision,
and F1 scores are: RBERT = 1 |x| X xi∈x max ˆ xj∈ˆ x x⊤ i ˆ xj , PBERT = 1 |ˆ x| X ˆ xj∈ˆ
x max xi∈x x⊤ i ˆ xj , FBERT = 2 PBERT · RBERT PBERT + RBERT .


In [27]:
# Cell 10: Attach all three needles to sample and print all fields
sample['needle_bm25']       = bm25_result
sample['needle_biencoder']  = biencoder_result
sample['needle_llm']        = llm_result

DIVIDER = "━" * 70

def pp(label, value, truncate=None):
    print(f"\n{DIVIDER}")
    print(f"  {label}")
    print(f"{DIVIDER}")
    if isinstance(value, dict):
        for k, v in value.items():
            v_str = str(v)
            if truncate and k == "text" and len(v_str) > truncate:
                v_str = v_str[:truncate] + f"  ... [{len(v_str)-truncate} chars truncated]"
            print(f"  {k}: {v_str}")
    elif isinstance(value, str) and truncate and len(value) > truncate:
        print(textwrap.fill(value[:truncate], width=88))
        print(f"  ... [{len(value)-truncate} chars truncated]")
    else:
        print(textwrap.fill(str(value), width=88))

pp("_id",             sample['_id'])
pp("domain",          sample['domain'])
pp("sub_domain",      sample['sub_domain'])
pp("difficulty",      sample['difficulty'])
pp("length",          sample['length'])
pp("question",        sample['question'])
pp("choice_A",        sample['choice_A'])
pp("choice_B",        sample['choice_B'])
pp("choice_C",        sample['choice_C'])
pp("choice_D",        sample['choice_D'])
pp("answer",          sample['answer'])
pp("context",         sample['context'], truncate=500)
pp("needle_bm25",     sample['needle_bm25'])
pp("needle_biencoder",sample['needle_biencoder'])
pp("needle_llm",      sample['needle_llm'])


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  _id
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
66f53892821e116aacb33225

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  domain
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Single-Document QA

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  sub_domain
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Academic

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  difficulty
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
easy

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  length
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
short

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  question
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Which of the foll

In [ ]:
# # Cell 11*: Needle position summary
# def needle_depth_pct(needle_text, context):
#     anchor = needle_text[:80].strip()
#     idx = context.find(anchor)
#     return round(idx / len(context) * 100, 1) if idx != -1 else None

# print(f"{'Method':<20} {'Depth %':>10}   Sentence indices")
# print("-" * 60)
# for result in [bm25_result, biencoder_result, llm_result]:
#     depth = needle_depth_pct(result["text"], sample["context"])
#     depth_str = f"{depth}%" if depth is not None else "not found"
#     print(f"{result['method']:<20} {depth_str:>10}   {result['sentence_indices']}")

Method                  Depth %   Sentence indices
------------------------------------------------------------
BM25Okapi                  5.9%   [69, 115, 493]
multi-qa-MiniLM-L6-cos-v1       1.1%   [12, 100, 284]
LLM-guided (claude-sonnet-4-6)      15.0%   [114, 115, 117]


In [39]:
# Cell 11: Needle position summary (fixed)

def needle_depth_pct(sentence_indices, all_sentences, context):
    """
    Find depth using the first sentence of the needle individually,
    since the joined needle string may not appear verbatim in the context
    due to whitespace/newline differences from NLTK tokenization.
    """
    if not sentence_indices:
        return None
    first_sentence = all_sentences[sentence_indices[0]].strip()
    # Try progressively shorter anchors in case of minor tokenization artifacts
    for anchor_len in [80, 50, 30]:
        anchor = first_sentence[:anchor_len].strip()
        idx = context.find(anchor)
        if idx != -1:
            return round(idx / len(context) * 100, 1)
    return None

print(f"{'Method':<36} {'Depth %':>10}   Sentence indices")
print("-" * 70)
for result in [bm25_result, biencoder_result, llm_result]:
    depth = needle_depth_pct(result["sentence_indices"], context_sentences, sample["context"])
    depth_str = f"{depth}%" if depth is not None else "not found"
    print(f"{result['method']:<36} {depth_str:>10}   {result['sentence_indices']}")

Method                                  Depth %   Sentence indices
----------------------------------------------------------------------
BM25Okapi                                  5.9%   [69, 115, 493]
multi-qa-MiniLM-L6-cos-v1                  1.1%   [12, 100, 284]
LLM-guided (claude-sonnet-4-6)            15.0%   [114, 115, 117]


In [40]:
# Cell 12: Needle position perturbation
#
# Uses the LLM-guided needle (best quality).
# Steps:
#   1. Remove needle sentences from the original sentence list
#   2. Reinsert the needle at 10% / 50% / 90% of the remaining sentence count
#   3. Join sentences back into a single context string
#   4. Store as needle_context1/2/3 plus needle as standalone entry

def remove_needle_from_context(all_sentences, needle_indices):
    """Return sentences with the needle indices removed, preserving order."""
    needle_set = set(needle_indices)
    return [s for i, s in enumerate(all_sentences) if i not in needle_set]

def insert_needle_at_depth(remaining_sentences, needle_text, depth_pct):
    """
    Insert needle_text as a block at approximately depth_pct (0–1)
    of the remaining sentence list. Returns a single joined string.
    """
    insert_at = int(len(remaining_sentences) * depth_pct)
    insert_at = max(0, min(insert_at, len(remaining_sentences)))  # clamp

    before  = remaining_sentences[:insert_at]
    after   = remaining_sentences[insert_at:]

    combined = before + [needle_text] + after
    return " ".join(combined)

def actual_depth_pct(context_str, needle_text):
    """Verify where the needle actually landed as a % of the final context."""
    anchor = needle_text[:80].strip()
    idx = context_str.find(anchor)
    return round(idx / len(context_str) * 100, 1) if idx != -1 else None


# ── Use LLM-guided needle ─────────────────────────────────────────────────────
needle_indices = llm_result['sentence_indices']
needle_text    = llm_result['text']

remaining_sentences = remove_needle_from_context(context_sentences, needle_indices)

print(f"Original sentence count : {len(context_sentences)}")
print(f"Needle sentence count   : {len(needle_indices)}")
print(f"Remaining sentence count: {len(remaining_sentences)}")
print(f"\nNeedle text:\n{textwrap.fill(needle_text, width=90)}")

Original sentence count : 684
Needle sentence count   : 3
Remaining sentence count: 681

Needle text:
BERTSCORE The complete score matches each token in x to a token in ˆ x to compute recall,
and each token in ˆ x to a token in x to compute precision. We use greedy matching to
maximize the matching similarity score,2 where each token is matched to the most similar
token in the other sentence. For a reference x and candidate ˆ x, the recall, precision,
and F1 scores are: RBERT = 1 |x| X xi∈x max ˆ xj∈ˆ x x⊤ i ˆ xj , PBERT = 1 |ˆ x| X ˆ xj∈ˆ
x max xi∈x x⊤ i ˆ xj , FBERT = 2 PBERT · RBERT PBERT + RBERT .


In [41]:
# Cell 13: Build the three perturbed contexts and update sample

POSITIONS = {
    "needle_context1": 0.10,
    "needle_context2": 0.50,
    "needle_context3": 0.90,
}

for field_name, depth in POSITIONS.items():
    context_str = insert_needle_at_depth(remaining_sentences, needle_text, depth)
    actual_depth = actual_depth_pct(context_str, needle_text)
    sample[field_name] = {
        "context":        context_str,
        "target_depth":   f"{int(depth*100)}%",
        "actual_depth":   f"{actual_depth}%" if actual_depth is not None else "not found",
        "needle_indices_removed": needle_indices,
    }
    print(f"{field_name} | target: {int(depth*100)}% | actual: {actual_depth}% "
          f"| chars: {len(context_str)}")

# ── Add needle as its own top-level entry ─────────────────────────────────────
sample['needle'] = {
    "text":             needle_text,
    "sentence_indices": needle_indices,
    "reasoning":        llm_result['reasoning'],
    "method":           llm_result['method'],
}

print(f"\nSample now has fields: {list(sample.keys())}")

needle_context1 | target: 10% | actual: 5.9% | chars: 134713
needle_context2 | target: 50% | actual: 35.3% | chars: 134713
needle_context3 | target: 90% | actual: 57.5% | chars: 134713

Sample now has fields: ['_id', 'domain', 'sub_domain', 'difficulty', 'length', 'question', 'choice_A', 'choice_B', 'choice_C', 'choice_D', 'answer', 'context', 'needle_bm25', 'needle_biencoder', 'needle_llm', 'needle_context1', 'needle_context2', 'needle_context3', 'needle']


In [45]:
# Cell 14 (revised): Build and output the final clean sample JSON

import json
import numpy as np

CONTEXT_PREVIEW_CHARS = 300

class NumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.integer): return int(obj)
        if isinstance(obj, np.floating): return float(obj)
        if isinstance(obj, np.ndarray): return obj.tolist()
        return super().default(obj)

KEEP_FIELDS = [
    "_id", "domain", "sub_domain", "difficulty", "length",
    "question", "choice_A", "answer", "context",
    "needle", "needle_context1", "needle_context2", "needle_context3",
]

final_sample = {
    **{k: sample[k] for k in KEEP_FIELDS if k in sample and k not in
       ("needle", "needle_context1", "needle_context2", "needle_context3")},
    "needle":         sample["needle"]["text"],
    "needle_context1": sample["needle_context1"]["context"],
    "needle_context2": sample["needle_context2"]["context"],
    "needle_context3": sample["needle_context3"]["context"],
}

# ── Preview with truncated context fields ────────────────────────────────────
def make_printable(obj, truncate_keys, limit):
    if isinstance(obj, dict):
        return {k: make_printable(v, truncate_keys, limit) for k, v in obj.items()}
    if isinstance(obj, list):
        return [make_printable(i, truncate_keys, limit) for i in obj]
    if isinstance(obj, str) and any(obj == final_sample.get(k) for k in truncate_keys) and len(obj) > limit:
        return obj[:limit] + f"  ... [{len(obj) - limit} chars truncated]"
    return obj

TRUNCATE_KEYS = {"context", "needle", "needle_context1", "needle_context2", "needle_context3"}
printable = {
    k: (v[:CONTEXT_PREVIEW_CHARS] + f"  ... [{len(v) - CONTEXT_PREVIEW_CHARS} chars truncated]"
        if k in TRUNCATE_KEYS and isinstance(v, str) and len(v) > CONTEXT_PREVIEW_CHARS
        else v)
    for k, v in final_sample.items()
}

print(json.dumps(printable, indent=2, cls=NumpyEncoder))

{
  "_id": "66f53892821e116aacb33225",
  "domain": "Single-Document QA",
  "sub_domain": "Academic",
  "difficulty": "easy",
  "length": "short",
  "question": "Which of the following methods is used in this article to calculate the precision similarity of two documents?",
  "choice_A": "After converting the document content to bert embedding, calculate the n-grams that occur in the reference and candidate",
  "answer": "B",
  "context": "BERTSCORE: EVALUATING TEXT GENERATION WITH\nBERT\nABSTRACT\nWe propose BERTSCORE, an automatic evaluation metric for text generation.\nAnalogously to common metrics, BERTSCORE computes a similarity score for\neach token in the candidate sentence with each token in the reference sentence.\nHowever, instead  ... [135613 chars truncated]",
  "needle": "BERTSCORE\nThe complete score matches each token in x to a token in \u02c6\nx to compute recall,\nand each token in \u02c6\nx to a token in x to compute precision. We use greedy matching to maximize\nthe m

# process whole dataset


In [48]:
# Cell 16 (revised): Process samples filtered by LENGTH parameter
#
# Set LENGTH to "short", "medium", "long", or None to process all 503.

import json
import time
import numpy as np
from tqdm.auto import tqdm

# ── Filter parameter ──────────────────────────────────────────────────────────
LENGTH_FILTER = "short"    # "short" | "medium" | "long" | None

SAVE_EVERY  = 25
RETRY_DELAY = 10
MAX_RETRIES = 3

# Output paths are named by filter so files don't overwrite each other
suffix       = LENGTH_FILTER if LENGTH_FILTER else "all"
OUTPUT_PATH  = f"longbench_v2_processed_{suffix}.json"
FAILED_PATH  = f"longbench_v2_failed_{suffix}.json"

KEEP_FIELDS_BASE = [
    "_id", "domain", "sub_domain", "difficulty", "length",
    "question", "choice_A", "answer", "context",
]

class NumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.integer): return int(obj)
        if isinstance(obj, np.floating): return float(obj)
        if isinstance(obj, np.ndarray): return obj.tolist()
        return super().default(obj)

def process_one_sample(row, api_key):
    answer_letter = row["answer"]
    answer_text   = row[f"choice_{answer_letter}"]

    sentences = split_into_sentences(row["context"])
    if len(sentences) == 0:
        raise ValueError("No sentences extracted from context.")

    llm, last_err = None, None
    for attempt in range(MAX_RETRIES):
        try:
            llm = extract_needle_llm(
                sentences   = sentences,
                question    = row["question"],
                answer_text = answer_text,
                api_key     = api_key,
            )
            break
        except Exception as e:
            last_err = e
            if attempt < MAX_RETRIES - 1:
                time.sleep(RETRY_DELAY * (attempt + 1))
    if llm is None:
        raise RuntimeError(f"LLM extraction failed after {MAX_RETRIES} attempts: {last_err}")

    needle_text    = llm["text"]
    needle_indices = llm["sentence_indices"]

    remaining = remove_needle_from_context(sentences, needle_indices)
    if len(remaining) == 0:
        raise ValueError("Remaining sentences empty after removing needle.")

    out = {k: row[k] for k in KEEP_FIELDS_BASE}
    out["needle"]          = needle_text
    out["needle_context1"] = insert_needle_at_depth(remaining, needle_text, 0.10)
    out["needle_context2"] = insert_needle_at_depth(remaining, needle_text, 0.50)
    out["needle_context3"] = insert_needle_at_depth(remaining, needle_text, 0.90)
    return out


# ── Filter dataset ────────────────────────────────────────────────────────────
all_rows = df.to_dict(orient="records")

if LENGTH_FILTER:
    filtered_rows = [r for r in all_rows if r["length"].lower() == LENGTH_FILTER.lower()]
else:
    filtered_rows = all_rows

print(f"LENGTH_FILTER : {LENGTH_FILTER or 'None (all samples)'}")
print(f"Total in dataset  : {len(all_rows)}")
print(f"Samples to process: {len(filtered_rows)}")
print(f"Output → {OUTPUT_PATH}")

# ── Length breakdown of the filtered set ─────────────────────────────────────
from collections import Counter
breakdown = Counter(r["length"] for r in filtered_rows)
print(f"Length breakdown  : {dict(breakdown)}")


# ── Main loop ─────────────────────────────────────────────────────────────────
processed, failed = [], []

for i, row in enumerate(tqdm(filtered_rows, desc=f"Processing [{suffix}]")):
    try:
        result = process_one_sample(row, ANTHROPIC_API_KEY)
        processed.append(result)
    except Exception as e:
        failed.append({"_id": row.get("_id", f"row_{i}"), "error": str(e), "index": i})
        tqdm.write(f"  [FAILED] index={i}  id={row.get('_id')}  —  {e}")

    if (i + 1) % SAVE_EVERY == 0:
        with open(OUTPUT_PATH, "w") as f:
            json.dump(processed, f, indent=2, cls=NumpyEncoder)
        tqdm.write(f"  [checkpoint] {len(processed)} processed, {len(failed)} failed")

# ── Final save ────────────────────────────────────────────────────────────────
with open(OUTPUT_PATH, "w") as f:
    json.dump(processed, f, indent=2, cls=NumpyEncoder)

with open(FAILED_PATH, "w") as f:
    json.dump(failed, f, indent=2)

print(f"\nDone.")
print(f"  Processed : {len(processed)} / {len(filtered_rows)}")
print(f"  Failed    : {len(failed)} / {len(filtered_rows)}")
print(f"  Output    : {OUTPUT_PATH}")
print(f"  Failed log: {FAILED_PATH}")

LENGTH_FILTER : short
Total in dataset  : 503
Samples to process: 180
Output → longbench_v2_processed_short.json
Length breakdown  : {'short': 180}


Processing [short]:   1%|          | 1/180 [00:37<1:50:29, 37.03s/it]

  [FAILED] index=0  id=66f36490821e116aacb2cc22  —  LLM extraction failed after 3 attempts: Error code: 429 - {'type': 'error', 'error': {'type': 'rate_limit_error', 'message': "This request would exceed your organization's rate limit of 30,000 input tokens per minute (org: 91bcb7bf-a0db-4cab-905e-9a2ec4529e71, model: claude-sonnet-4-6). For details, refer to: https://docs.claude.com/en/api/rate-limits. You can see the response headers for current usage. Please reduce the prompt length or the maximum tokens requested, or try again later. You may also contact sales at https://claude.com/contact-sales to discuss your options for a rate limit increase."}, 'request_id': 'req_011CaBxPkWDuL6M5AKyX8UjV'}


Processing [short]:   1%|          | 2/180 [01:13<1:49:40, 36.97s/it]

  [FAILED] index=1  id=66ebed525a08c7b9b35e1cb4  —  LLM extraction failed after 3 attempts: Error code: 429 - {'type': 'error', 'error': {'type': 'rate_limit_error', 'message': "This request would exceed your organization's rate limit of 30,000 input tokens per minute (org: 91bcb7bf-a0db-4cab-905e-9a2ec4529e71, model: claude-sonnet-4-6). For details, refer to: https://docs.claude.com/en/api/rate-limits. You can see the response headers for current usage. Please reduce the prompt length or the maximum tokens requested, or try again later. You may also contact sales at https://claude.com/contact-sales to discuss your options for a rate limit increase."}, 'request_id': 'req_011CaBxSTweSEcgNm3yGz2ux'}


Processing [short]:   1%|          | 2/180 [01:21<2:01:15, 40.87s/it]


KeyboardInterrupt: 

In [ ]:
# Cell 17 (revised): Inspect results

with open(OUTPUT_PATH) as f:
    results = json.load(f)

print(f"Samples in output file : {len(results)}")
print(f"Fields per sample      : {list(results[0].keys())}")
print(f"Length values present  : {set(r['length'] for r in results)}")
print(f"\nSample preview (first entry, contexts truncated to 200 chars):")

preview = {
    k: (v[:200] + f"  ... [{len(v)-200} chars]" if isinstance(v, str) and len(v) > 200 else v)
    for k, v in results[0].items()
}
print(json.dumps(preview, indent=2))

if failed:
    print(f"\nFailed samples ({len(failed)}):")
    for entry in failed:
        print(f"  index={entry['index']}  id={entry['_id']}  error={entry['error']}")

In [ ]:
import json
combined = []
for suffix in ["short", "medium", "long"]:
    with open(f"longbench_v2_processed_{suffix}.json") as f:
        combined.extend(json.load(f))
print(f"Combined total: {len(combined)}")
with open("longbench_v2_processed_all.json", "w") as f:
    json.dump(combined, f, indent=2)